# 03 — Model Training & Evaluation

End-to-end walkthrough:
1. Build dataloaders
2. Instantiate DCRNN
3. Run a short training loop (or load a checkpoint)
4. Evaluate on test set — MAE / RMSE / MAPE per horizon step
5. Visualise predictions vs ground truth on selected sensors

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import torch

from config import MODEL_DIR, cfg
from src.dataset import build_dataloaders
from src.model import build_model
from src.utils import (
    compute_all_metrics,
    load_checkpoint,
    masked_mae,
    resolve_device,
    seed_everything,
)

seed_everything(cfg.train.seed)
device = resolve_device(cfg.train.device)
print(f'Device: {device}')

## 1. Build dataloaders

In [ ]:
train_loader, val_loader, test_loader, scaler, adj_mx, sensor_ids = build_dataloaders(cfg.data)

print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')
print(f'Test  batches: {len(test_loader)}')

x, y = next(iter(train_loader))
print(f'x shape: {x.shape}  (B, T, N, F)')
print(f'y shape: {y.shape}  (B, H, N, F)')

## 2. Instantiate DCRNN

In [ ]:
model = build_model(adj_mx).to(device)
print(f'Trainable parameters: {model.count_parameters():,}')

## 3. Load checkpoint (or train a few epochs)

In [ ]:
ckpt_path = MODEL_DIR / 'dcrnn_best.pt'

if ckpt_path.exists():
    ckpt = load_checkpoint(ckpt_path, model, device=device)
    print(f"Loaded checkpoint from epoch {ckpt['epoch']}")
    print(f"Stored val MAE: {ckpt['metrics'].get('val_MAE', 'N/A')}")
else:
    print('No checkpoint found — running 3 warm-up epochs …')
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for ep in range(3):
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb[..., :1].to(device)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb, targets=yb)
            loss = masked_mae(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            losses.append(loss.item())
        print(f'  Epoch {ep+1}/3  loss={np.mean(losses):.4f}')

## 4. Per-horizon-step evaluation

In [ ]:
model.eval()
H = cfg.data.horizon

mae_per_h  = np.zeros(H)
rmse_per_h = np.zeros(H)
mape_per_h = np.zeros(H)
n_batches  = 0

from src.utils import masked_mae, masked_rmse, masked_mape

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        yb = yb[..., :1].to(device)
        pred = model(xb)                          # (B, H, N, 1)
        pred_d = scaler.inverse_transform(pred)
        true_d = scaler.inverse_transform(yb)
        for h in range(H):
            mae_per_h[h]  += masked_mae(pred_d[:, h], true_d[:, h]).item()
            rmse_per_h[h] += masked_rmse(pred_d[:, h], true_d[:, h]).item()
            mape_per_h[h] += masked_mape(pred_d[:, h], true_d[:, h]).item()
        n_batches += 1

mae_per_h  /= n_batches
rmse_per_h /= n_batches
mape_per_h /= n_batches

steps = np.arange(1, H + 1) * 5   # minutes

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, values, label, color in [
    (axes[0], mae_per_h,  'MAE',      'steelblue'),
    (axes[1], rmse_per_h, 'RMSE',     'coral'),
    (axes[2], mape_per_h, 'MAPE (%)', 'seagreen'),
]:
    ax.plot(steps, values, marker='o', color=color)
    ax.set_xlabel('Horizon (minutes)')
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs Horizon')
    ax.set_xticks(steps[::3])

plt.tight_layout()
plt.show()

print(f'15-min MAE: {mae_per_h[2]:.3f}   RMSE: {rmse_per_h[2]:.3f}   MAPE: {mape_per_h[2]:.2f}%')
print(f'60-min MAE: {mae_per_h[-1]:.3f}   RMSE: {rmse_per_h[-1]:.3f}   MAPE: {mape_per_h[-1]:.2f}%')

## 5. Prediction vs ground truth — single sensor

In [ ]:
SENSOR_IDX = 42   # ← change to inspect a different sensor
N_WINDOWS  = 96   # number of consecutive windows to visualise (~8 h)

all_pred, all_true = [], []
model.eval()

with torch.no_grad():
    for i, (xb, yb) in enumerate(test_loader):
        if i * cfg.data.batch_size >= N_WINDOWS:
            break
        xb = xb.to(device)
        yb = yb[..., :1].to(device)
        pred = model(xb)
        pred_d = scaler.inverse_transform(pred)
        true_d = scaler.inverse_transform(yb)
        # Take first step of horizon, sensor SENSOR_IDX
        all_pred.append(pred_d[:, 0, SENSOR_IDX, 0].cpu().numpy())
        all_true.append(true_d[:, 0, SENSOR_IDX, 0].cpu().numpy())

pred_arr = np.concatenate(all_pred)[:N_WINDOWS]
true_arr = np.concatenate(all_true)[:N_WINDOWS]
time_ax  = np.arange(len(pred_arr)) * 5  # minutes

plt.figure(figsize=(16, 4))
plt.plot(time_ax, true_arr, label='Ground truth', lw=1.5, color='steelblue')
plt.plot(time_ax, pred_arr, label='Prediction',   lw=1.5, color='coral', ls='--')
plt.fill_between(time_ax,
                 np.minimum(pred_arr, true_arr),
                 np.maximum(pred_arr, true_arr),
                 alpha=0.2, color='orange')
plt.xlabel('Time (minutes)')
plt.ylabel('Traffic speed (mph)')
plt.title(f'Sensor {sensor_ids[SENSOR_IDX]} — 15-min-ahead Prediction')
plt.legend()
plt.tight_layout()
plt.show()

mae = np.abs(pred_arr - true_arr).mean()
print(f'Sensor {sensor_ids[SENSOR_IDX]}  MAE over {len(pred_arr)} steps: {mae:.3f} mph')